In [ ]:
import shutil, os

# 캐시 삭제
shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

# 캐시 삭제 확인
assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

# 환경변수 설정
os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

# 환경변수 확인
assert os.environ.get("UNSLOTH_TARGET_GB") == "4", f"UNSLOTH_TARGET_GB 설정 실패: {os.environ.get('UNSLOTH_TARGET_GB')}"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")
from unsloth import FastModel
import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")


In [1]:
"""
Qwen3.5-0.8B LoRA Fine-tuning (Unsloth)
RTX 6000 Pro Blackwell (96GB) / chi 환경
"""

import shutil, os

# 캐시 삭제
shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

# 캐시 삭제 확인
assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

# 환경변수 설정
os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

# 환경변수 확인
assert os.environ.get("UNSLOTH_TARGET_GB") == "4", f"UNSLOTH_TARGET_GB 설정 실패: {os.environ.get('UNSLOTH_TARGET_GB')}"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")
from unsloth import FastModel

# unsloth 내부에서 실제로 읽히는지 확인
import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch

from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()

# -------------------------
# Config
# -------------------------
MODEL_ID = "unsloth/Qwen3.5-0.8B"
DATASET_PATH = "./data/5_qwen_dataset.jsonl"
OUTPUT_DIR = "./model/qwen3.5-0.8b-lora"

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
LORA_RANK = 16
LORA_ALPHA = 32
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 모델 + 토크나이저
# -------------------------
print("📦 모델 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",

)

# -------------------------
# LoRA
# -------------------------
print("🔧 LoRA 어댑터 부착...")
model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,
)
model.print_trainable_parameters()


# -------------------------
# 데이터셋
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# 내부 토크나이저 참조 (Qwen3.5는 Processor 반환)
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def format_messages(example):
    text = _tok.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


dataset = dataset.map(format_messages, remove_columns=dataset.column_names, num_proc=32)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")


# -------------------------
# 토큰 길이 초과 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    example["n_tokens"] = len(_tok.encode(example["text"]))
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")

# -------------------------
# 학습
# -------------------------
print("🚀 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

trainer.train()

# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

✅ 컴파일 캐시 삭제 확인 완료
✅ UNSLOTH_TARGET_GB = 4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ unsloth TARGET_GB = 4
📦 모델 로딩...
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.936 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 2923.88it/s]


🔧 LoRA 어댑터 부착...
Unsloth: Making `model.base_model.model.model.language_model` require gradients
trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435
📂 데이터셋 로딩...


...

✅ 필터: 66400개 → 66305개 (95개 제외)
🚀 학습 시작...


Unsloth: Tokenizing ["text"] (num_proc=32): 100%|██████████| 66305/66305 [02:25<00:00, 454.49 examples/s] 
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66,305 | Num Epochs = 1 | Total steps = 8,289
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 6,389,760 of 859,375,680 (0.74% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.255530
20,0.234763
30,0.259916
40,0.208142
50,0.292130
60,0.256292
70,0.274309
80,0.158570
90,0.258970
100,0.181017


💾 어댑터 저장...
✅ 완료! 저장 위치: ./model/qwen3.5-0.8b-lora
